# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import nltk

!pip install -qqq chromadb
!pip install -qqq rouge-score

# Download necessary NLTK data (if not already downloaded)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError: # Catch LookupError as indicated in the traceback
    nltk.download('punkt')

print("Installed chromadb, rouge-score, and downloaded NLTK punkt tokenizer.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Installed chromadb, rouge-score, and downloaded NLTK punkt tokenizer.


In [ ]:
import chromadb

client = chromadb.PersistentClient(path=chroma_db_dir)

for c in client.list_collections():
    col = client.get_collection(c.name)
    print(f"{c.name}: {col.count()} documents")


sop_collection: 0 documents
langchain: 64 documents


In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import json

# Define paths based on user input
base_data_dir = '/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/'
policies_dir = os.path.join(base_data_dir, 'sop_documents')
chroma_db_dir = os.path.join(base_data_dir, 'chroma_db')
df_test_path = os.path.join(base_data_dir, 'df_test.csv')

# Load df_test.csv
try:
    df_test = pd.read_csv(df_test_path)
    print(f"Loaded df_test.csv with {len(df_test)} rows.")
except FileNotFoundError:
    print(f"Error: df_test.csv not found at {df_test_path}. Please ensure it exists.")
    df_test = pd.DataFrame() # Create an empty DataFrame to avoid errors later

# Set device for model loading
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load LLM and Tokenizer
llm_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {llm_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, # Use bfloat16 for GPU, float32 for CPU
    device_map=device
)

# Initialize text generation pipeline
generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device=device
)

# Helper function for text generation, handling prompt removal
def generate_text(prompt: str, max_new_tokens: int = 100) -> str:
    try:
        output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]['generated_text']
        # The pipeline output typically includes the prompt. Need to strip it.
        if output.startswith(prompt):
            return output[len(prompt):].strip()
        return output.strip() # Fallback if prompt isn't directly at start.
    except Exception as e:
        print(f"Error during text generation: {e}")
        return "[Generation Error]"

# Load Embedding Model
embedding_model_name = "all-MiniLM-L6-v2"
print(f"Loading Embedding Model: {embedding_model_name}...")
embedding_model = SentenceTransformer(embedding_model_name)

# Initialize ChromaDB Client and get collection
print(f"Loading ChromaDB from {chroma_db_dir}...")
chroma_client = chromadb.PersistentClient(path=chroma_db_dir)
vector_store = chroma_client.get_or_create_collection("langchain")
print(f"ChromaDB collection '{vector_store.name}' loaded with {vector_store.count()} documents.")

# Define generate_baseline function
def generate_baseline(query: str) -> str:
    prompt = f"Answer the following question: {query}"
    return generate_text(prompt, max_new_tokens=100)

# Define generate_naive_rag function
def generate_naive_rag(query: str, k: int = 3) -> str:
    # Retrieve relevant documents
    query_embedding = embedding_model.encode(query).tolist()
    results = vector_store.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=['documents']
    )
    context_docs = [doc for sublist in results['documents'] for doc in sublist]
    context = "\n".join(context_docs)

    # Construct RAG prompt
    rag_prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"
    return generate_text(rag_prompt, max_new_tokens=200)

print("Shared setup complete. Helper functions generate_baseline and generate_naive_rag are defined.")

Loaded df_test.csv with 391 rows.
Using device: cuda
Loading LLM: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading Embedding Model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading ChromaDB from /content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/chroma_db...
ChromaDB collection 'langchain' loaded with 64 documents.
Shared setup complete. Helper functions generate_baseline and generate_naive_rag are defined.


In [ ]:
query = "How do I reset my password?"

query_embedding = embedding_model.encode(query).tolist()

results = vector_store.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents"]
)

print(results["documents"])

[['# Password Reset\n\n## Scope\nThis covers the routine case of a customer who knows their account email but\nhas forgotten or wants to change their password. Lockouts, lost email access,\nor suspected compromise follow the account recovery procedure instead.', '## Self-Service Flow\nDirect the customer to the "Forgot password" link on the sign-in page. A reset\nemail is sent to the registered address and the link is valid for 60 minutes\nand single use. Advise checking spam folders and allowing a few minutes for\ndelivery.', '# Account Recovery\n\n## When To Use\nUse this procedure when a customer cannot access their account due to a\nlockout, a lost email address, suspected unauthorized access, or a disabled\naccount — anything beyond a simple forgotten password (see the password reset\nprocedure for that case).']]


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [ ]:
# YOUR CODE HERE
print(df_test.columns.tolist())


['flags', 'instruction', 'category', 'intent', 'response', 'chatml_instruction', 'input_ids', 'labels', 'attention_mask']


In [ ]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm

# ==========================================================
# File paths
# ==========================================================
base_data_dir = "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset"

outputs_json_path = os.path.join(base_data_dir, "outputs.json")
csv_path = os.path.join(base_data_dir, "automated_test_outputs.csv")

# ==========================================================
# Automated Testing
# ==========================================================
all_outputs = []

print("Executing automated testing for Baseline and Naive RAG...")

for _, row in tqdm(
    df_test.iterrows(),
    total=len(df_test),
    desc="Generating outputs"
):

    # User query
    query = row["instruction"]

    # ----------------------------
    # Baseline Generation
    # ----------------------------
    try:
        baseline_output = generate_baseline(query)
    except Exception as e:
        baseline_output = f"[ERROR] {str(e)}"

    # ----------------------------
    # Naive RAG Generation
    # ----------------------------
    try:
        naive_rag_output = generate_naive_rag(query)
    except Exception as e:
        naive_rag_output = f"[ERROR] {str(e)}"

    # ----------------------------
    # Store outputs
    # ----------------------------
    all_outputs.append({
        "instruction": query,
        "category": row["category"],
        "intent": row["intent"],
        "ground_truth": row["response"],
        "baseline_output": baseline_output,
        "naive_rag_output": naive_rag_output
    })

    # ----------------------------
    # Save checkpoint every 10 queries
    # ----------------------------
    if len(all_outputs) % 10 == 0:

        checkpoint_df = pd.DataFrame(all_outputs)

        with open(outputs_json_path, "w") as f:
            json.dump(all_outputs, f, indent=4)

        checkpoint_df.to_csv(csv_path, index=False)

        print(f"Checkpoint saved ({len(all_outputs)} queries).")

# ==========================================================
# Final Save
# ==========================================================
results_df = pd.DataFrame(all_outputs)

with open(outputs_json_path, "w") as f:
    json.dump(all_outputs, f, indent=4)

results_df.to_csv(csv_path, index=False)

print("\nAutomated testing complete.")
print(f"Generated outputs for {len(results_df)} queries.")

print("\nFiles saved successfully:")
print(f"JSON : {outputs_json_path}")
print(f"CSV  : {csv_path}")

# Preview results
display(results_df.head())

Executing automated testing for Baseline and Naive RAG...


Generating outputs:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Checkpoint saved (10 queries).


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

...

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Checkpoint saved (390 queries).


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Automated testing complete.
Generated outputs for 391 queries.

Files saved successfully:
JSON : /content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/outputs.json
CSV  : /content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/automated_test_outputs.csv


,instruction,category,intent,ground_truth,baseline_output,naive_rag_output
0,need help to shop several of ur item,ORDER,place_order,Thank you for expressing your interest in shop...,in one order? I'm a new customer and don't kno...,"I'm sorry, but as an AI language model, I don'..."
1,i need assistance trying to update the shippin...,SHIPPING,change_shipping_address,I'll take care of it! I'm here to assist you w...,in my account. can you help me? Answer accordi...,To assist you in updating the shipping address...
2,I do not know how to check the cancellation p...,CANCEL,check_cancellation_fee,No problem! To check the cancellation penaltie...,for a flight. Where can I find this informatio...,"To check the cancellation penalties, you can r..."
3,want assistance to see the payment modalities,PAYMENT,check_payment_methods,Assuredly! I'm here to assist you in exploring...,of a product or service? 1. call customer supp...,"I'm sorry, but as an AI language model, I don'..."
4,how can I get information about opening a stan...,ACCOUNT,create_account,Thank you for expressing interest in opening a...,Answer Choices: - go to the website - ask you...,"To open a standard account, you need to follow..."


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [ ]:
# Initialize counters
baseline_adherence_count = 0
naive_rag_adherence_count = 0

total_outputs = len(all_outputs)

print("Measuring format adherence...")

for output_data in all_outputs:

    # Baseline output
    baseline_output = output_data.get("baseline_output", "")

    if (
        isinstance(baseline_output, str)
        and baseline_output.strip()
        and not baseline_output.startswith("[ERROR]")
    ):
        baseline_adherence_count += 1

    # Naive RAG output
    naive_rag_output = output_data.get("naive_rag_output", "")

    if (
        isinstance(naive_rag_output, str)
        and naive_rag_output.strip()
        and not naive_rag_output.startswith("[ERROR]")
    ):
        naive_rag_adherence_count += 1

# Calculate percentages
baseline_adherence_rate = (
    baseline_adherence_count / total_outputs * 100
    if total_outputs else 0
)

naive_rag_adherence_rate = (
    naive_rag_adherence_count / total_outputs * 100
    if total_outputs else 0
)

print("\nFormat Adherence Results")
print("-" * 40)
print(
    f"Baseline : {baseline_adherence_rate:.2f}% "
    f"({baseline_adherence_count}/{total_outputs})"
)
print(
    f"Naive RAG: {naive_rag_adherence_rate:.2f}% "
    f"({naive_rag_adherence_count}/{total_outputs})"
)

Measuring format adherence...

Format Adherence Results
----------------------------------------
Baseline : 100.00% (391/391)
Naive RAG: 100.00% (391/391)


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [ ]:
# YOUR CODE HERE
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# ==========================================================
# 3.3.3 Measure Execution Success (ROUGE / BLEU)
# ==========================================================

import nltk
import pandas as pd
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

# Download tokenizer (only runs once)
nltk.download("punkt", quiet=True)

# Initialize scorers
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

smoothie = SmoothingFunction().method1

print("Calculating ROUGE-1, ROUGE-L and BLEU scores...")

for output_data in tqdm(all_outputs):

    # Ground-truth SOP response
    reference = output_data["ground_truth"]

    # Generated responses
    baseline = output_data["baseline_output"]
    rag = output_data["naive_rag_output"]

    # ------------------------------------------------------
    # Baseline Scores
    # ------------------------------------------------------
    if (
        isinstance(baseline, str)
        and baseline.strip()
        and not baseline.startswith("[ERROR]")
    ):

        rouge = scorer.score(reference, baseline)

        output_data["baseline_rouge1"] = rouge["rouge1"].fmeasure
        output_data["baseline_rougeL"] = rouge["rougeL"].fmeasure

        output_data["baseline_bleu"] = sentence_bleu(
            [word_tokenize(reference.lower())],
            word_tokenize(baseline.lower()),
            smoothing_function=smoothie
        )

    else:
        output_data["baseline_rouge1"] = 0.0
        output_data["baseline_rougeL"] = 0.0
        output_data["baseline_bleu"] = 0.0

    # ------------------------------------------------------
    # Naive RAG Scores
    # ------------------------------------------------------
    if (
        isinstance(rag, str)
        and rag.strip()
        and not rag.startswith("[ERROR]")
    ):

        rouge = scorer.score(reference, rag)

        output_data["naive_rag_rouge1"] = rouge["rouge1"].fmeasure
        output_data["naive_rag_rougeL"] = rouge["rougeL"].fmeasure

        output_data["naive_rag_bleu"] = sentence_bleu(
            [word_tokenize(reference.lower())],
            word_tokenize(rag.lower()),
            smoothing_function=smoothie
        )

    else:
        output_data["naive_rag_rouge1"] = 0.0
        output_data["naive_rag_rougeL"] = 0.0
        output_data["naive_rag_bleu"] = 0.0

print("\nROUGE/BLEU calculation complete.")

# ==========================================================
# Summary Table
# ==========================================================

results_df = pd.DataFrame(all_outputs)

summary = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-L",
        "BLEU"
    ],
    "Baseline": [
        results_df["baseline_rouge1"].mean(),
        results_df["baseline_rougeL"].mean(),
        results_df["baseline_bleu"].mean()
    ],
    "Naive RAG": [
        results_df["naive_rag_rouge1"].mean(),
        results_df["naive_rag_rougeL"].mean(),
        results_df["naive_rag_bleu"].mean()
    ]
})

summary = summary.round(4)

print("\nAverage Scores")
display(summary)

Calculating ROUGE-1, ROUGE-L and BLEU scores...


  0%|          | 0/391 [00:00<?, ?it/s]


ROUGE/BLEU calculation complete.

Average Scores


,Metric,Baseline,Naive RAG
0,ROUGE-1,0.2801,0.3434
1,ROUGE-L,0.1532,0.1775
2,BLEU,0.0234,0.0420


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [ ]:
# ==========================================================
# 3.3.4 Measure Output Consistency
# ==========================================================

print("Measuring Output Consistency...")

# Select a sample query
if not df_test.empty:
    sample_query = df_test.iloc[0]["instruction"]
else:
    raise ValueError("df_test is empty.")

num_runs = 3

baseline_outputs = []
naive_rag_outputs = []

print(f"\nRunning the same query {num_runs} times...\n")
print(f"Query: {sample_query}\n")

# Generate outputs multiple times
for _ in range(num_runs):
    baseline_outputs.append(generate_baseline(sample_query))
    naive_rag_outputs.append(generate_naive_rag(sample_query))

# Check determinism
baseline_consistent = all(
    output == baseline_outputs[0]
    for output in baseline_outputs
)

naive_rag_consistent = all(
    output == naive_rag_outputs[0]
    for output in naive_rag_outputs
)

print("=" * 50)
print("Output Consistency Results")
print("=" * 50)

print(f"Baseline Consistent : {baseline_consistent}")
print(f"Naive RAG Consistent: {naive_rag_consistent}")

# Display outputs only if inconsistent
if not baseline_consistent:
    print("\nBaseline Outputs:")
    for i, output in enumerate(baseline_outputs, 1):
        print(f"\nRun {i}\n{'-'*20}")
        print(output)

if not naive_rag_consistent:
    print("\nNaive RAG Outputs:")
    for i, output in enumerate(naive_rag_outputs, 1):
        print(f"\nRun {i}\n{'-'*20}")
        print(output)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Measuring Output Consistency...

Running the same query 3 times...

Query: need help to shop several of ur item



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Output Consistency Results
Baseline Consistent : True
Naive RAG Consistent: True


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [ ]:
# YOUR CODE HERE
# ==========================================================
# 3.3.5 Measure Hallucination Frequency
# ==========================================================

import pandas as pd

results_df = pd.DataFrame(all_outputs)

# Threshold below which we consider the response hallucinated
HALLUCINATION_THRESHOLD = 0.20

# Count hallucinations
baseline_hallucinations = (
    results_df["baseline_rougeL"] < HALLUCINATION_THRESHOLD
).sum()

naive_rag_hallucinations = (
    results_df["naive_rag_rougeL"] < HALLUCINATION_THRESHOLD
).sum()

total = len(results_df)

baseline_frequency = baseline_hallucinations / total * 100
naive_rag_frequency = naive_rag_hallucinations / total * 100

print("=" * 50)
print("Hallucination Frequency")
print("=" * 50)

print(
    f"Baseline : {baseline_frequency:.2f}% "
    f"({baseline_hallucinations}/{total})"
)

print(
    f"Naive RAG: {naive_rag_frequency:.2f}% "
    f"({naive_rag_hallucinations}/{total})"
)

Hallucination Frequency
Baseline : 85.68% (335/391)
Naive RAG: 74.17% (290/391)


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [ ]:
# YOUR CODE HERE
# ==========================================================
# 3.4.1 Compare Baseline vs Naive RAG (Solution V1)
# ==========================================================

import pandas as pd

# Convert outputs to DataFrame if not already done
results_df = pd.DataFrame(all_outputs)

# ----------------------------------------------------------
# Functional Correctness (Average Scores)
# ----------------------------------------------------------
baseline_rouge1 = results_df["baseline_rouge1"].mean()
rag_rouge1 = results_df["naive_rag_rouge1"].mean()

baseline_rougeL = results_df["baseline_rougeL"].mean()
rag_rougeL = results_df["naive_rag_rougeL"].mean()

baseline_bleu = results_df["baseline_bleu"].mean()
rag_bleu = results_df["naive_rag_bleu"].mean()

# ----------------------------------------------------------
# Consistency
# ----------------------------------------------------------
baseline_consistency = 100 if baseline_consistent else 0
rag_consistency = 100 if naive_rag_consistent else 0

# ----------------------------------------------------------
# Hallucination Frequency
# ----------------------------------------------------------
baseline_hall = baseline_frequency
rag_hall = naive_rag_frequency

# ----------------------------------------------------------
# Percentage Improvement Function
# ----------------------------------------------------------
def pct_change(old, new):
    if old == 0:
        return float("nan")
    return ((new - old) / old) * 100

# ----------------------------------------------------------
# Summary Table
# ----------------------------------------------------------
comparison = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-L",
        "BLEU",
        "Consistency (%)",
        "Hallucination Frequency (%)"
    ],
    "Baseline": [
        baseline_rouge1,
        baseline_rougeL,
        baseline_bleu,
        baseline_consistency,
        baseline_hall
    ],
    "Naive RAG": [
        rag_rouge1,
        rag_rougeL,
        rag_bleu,
        rag_consistency,
        rag_hall
    ]
})

# Improvement %
comparison["% Change"] = comparison.apply(
    lambda row: pct_change(row["Baseline"], row["Naive RAG"]),
    axis=1
)

# Hallucination is better when LOWER
comparison.loc[
    comparison["Metric"] == "Hallucination Frequency (%)",
    "% Change"
] *= -1

# Round for display
comparison = comparison.round(4)

print("=" * 70)
print("Baseline vs Naive RAG")
print("=" * 70)

display(comparison)



Baseline vs Naive RAG


,Metric,Baseline,Naive RAG,% Change
0,ROUGE-1,0.2801,0.3434,22.5976
1,ROUGE-L,0.1532,0.1775,15.8165
2,BLEU,0.0234,0.0420,79.5738
3,Consistency (%),100.0000,100.0000,0.0000
4,Hallucination Frequency (%),85.6777,74.1688,13.4328


---
## Save Artifacts

In [ ]:
# Save comparison table for Notebook 5 checklist

v1_metrics_path = "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/v1_metrics.csv"

comparison.to_csv(v1_metrics_path, index=False)

print("✓ v1_metrics.csv saved successfully!")
print(v1_metrics_path)

# Preview
display(comparison)
evaluation_results_path = "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/evaluation_results.csv"

results_df.to_csv(evaluation_results_path, index=False)

print("✓ evaluation_results.csv saved!")
print(evaluation_results_path)

✓ v1_metrics.csv saved successfully!
/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/v1_metrics.csv


,Metric,Baseline,Naive RAG,% Change
0,ROUGE-1,0.2801,0.3434,22.5976
1,ROUGE-L,0.1532,0.1775,15.8165
2,BLEU,0.0234,0.0420,79.5738
3,Consistency (%),100.0000,100.0000,0.0000
4,Hallucination Frequency (%),85.6777,74.1688,13.4328


✓ evaluation_results.csv saved!
/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/evaluation_results.csv


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**